# Au 30 keV — planar PSF simulation

Run the cells from top to bottom on your cluster GPU node. Change ordinary Python variables in the settings cell; no shell commands, argument parser, or command-line argument lists are used.

This notebook uses the transport engine and PSF analysis functions from **seemc-taichi v0.9.3** (also present in v0.9.1/v0.9.2). It requires your working Taichi installation, the matching `seemc-imaging` reference loader, NumPy, and Matplotlib.

The test starts with **Au, 30,000 eV, 0°, 1,000 primaries**, one batch of 1,000, capacity 10,000,000, CUDA, and **f64**, matching your latest successful cluster run. Change `precision` to `'f32'` to match the original command. The simulation is a point beam; optional Gaussian beam convolution is applied only during analysis.

Initialization is part of the notebook. It resets Taichi's fields and kernels, so complete any other active Taichi calculations before rerunning the initialization cell. After changing simulation settings, rerun from initialization onward. After changing only analysis settings, rerun the analysis and display cells.


## 1. Select your working Taichi installation


In [ ]:
import sys
from pathlib import Path

taichi_root = Path('/home/fs72041/olgaridzel/dev/taichi_gpu_173')
if not (taichi_root / 'taichi' / '__init__.py').is_file():
    raise FileNotFoundError(f'Check the working Taichi installation: {taichi_root}')
sys.path.insert(0, str(taichi_root))

# Optional: use a source checkout if seemc-taichi is not installed in this kernel.
# Change this path if your project is elsewhere.
seemc_project = Path('/home/fs72041/olgaridzel/dev/seemc-taichi-v0.9.3')
if (seemc_project / 'src' / 'seemc_taichi').is_dir():
    sys.path.insert(1, str(seemc_project / 'src'))

import taichi as ti
print('Python:', sys.executable)
print('Taichi:', ti.__version__)
print('Loaded from:', ti.__file__)
if not Path(ti.__file__).resolve().is_relative_to(taichi_root.resolve()):
    raise RuntimeError('A different Taichi was already imported. Restart the kernel and run this cell first.')


## 2. Edit settings


In [ ]:
# Paths resolve relative to the notebook kernel's working directory.
database = Path('../MaterialDatabase.pkl').resolve()
output_prefix = Path('au_30keV_test')

# Simulation settings.
material = 'Au'
energy_ev = 30000.0
angle_deg = 0.0
arch = 'cuda'
precision = 'f64'                # Matches your latest successful cluster run; 'f32' is also supported.
fast_math = False
enable_fallback = False
n = 1000
batch_size = 1000
capacity = 10000000             # Per-batch capacity, matching your original test.
steps_per_chunk = 256
seed = 20260921

# Analysis settings. These can be changed without rerunning transport.
beam_sigma_nm = 0.0             # 0 = point response; 1 = Gaussian sigma of 1 nm.
convolution_seed = 20260921
r_min_nm = 0.02
r_max_nm = None                 # None = choose from the recorded radii.
radial_bins = 140
min_count_per_bin = 10
seed_bins = 90
cutoff_ev = None                # None = material's SE/BSE cutoff (usually 50 eV).

# Set True to analyze the events saved by your successful simulation.
# This skips Taichi initialization and transport; the NPZ is never overwritten.
analyze_existing = False
existing_events = Path('au_30keV_test_events.npz')

print('Working directory:', Path.cwd())
print('Material database:', database)
print('Output prefix:', output_prefix.resolve())


## 3. Import transport and analysis functions


In [ ]:
from dataclasses import asdict
import json
import math
import time

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from seemc_taichi import __version__
from seemc_taichi.config import BackendConfig
from seemc_taichi.plane import BulkPhysicsConfig, PlaneTransportEngine, load_reference_plane_tables
from seemc_taichi.surface import SurfacePhysicsConfig
from seemc_taichi.plane_psf import (
    CLASSIFICATION_DESCRIPTIONS, classify_emissions, emission_xy_nm, birth_radius_depth_nm,
    population_summary, radial_bin_edges, radial_profiles,
    seed_radius_depth_summary, write_profile_csv, write_rows_csv,
)

print('seemc-taichi:', __version__)
EVENT_KEYS = (
    "emission_energy_ev",
    "emission_x_angstrom",
    "emission_y_angstrom",
    "emission_z_angstrom",
    "emission_ux",
    "emission_uy",
    "emission_uz",
    "emission_electron_id",
    "emission_parent_id",
    "emission_root_primary_id",
    "emission_generation",
    "emission_is_cascade",
    "emission_mechanism",
    "emission_own_elastic_count",
    "emission_own_inelastic_count",
    "emission_path_elastic_count",
    "emission_path_inelastic_count",
    "emission_birth_x_angstrom",
    "emission_birth_y_angstrom",
    "emission_birth_z_angstrom",
)

def _save_archive_atomic(path: Path, **arrays):
    temporary = path.with_suffix(path.suffix + ".tmp")
    try:
        with temporary.open("wb") as stream:
            np.savez_compressed(stream, **arrays)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)


### Plot helpers: handle sparse or empty populations safely

These functions keep the original plot layouts and mark empty panels without inventing counts. The default bin-count threshold stays at 10; reduce it only if you want to see noisier bins.


In [ ]:
def _plot_psf_curve(
    ax,
    radius,
    profiles,
    name,
    label,
    color,
    *,
    min_count_per_bin,
    ls="-",
    lw=2.0,
):
    values = np.asarray(profiles[f"psf__{name}"], dtype=float)
    count = np.asarray(profiles[f"count__{name}"], dtype=np.int64)
    reliable = (
        (count >= int(min_count_per_bin)) & np.isfinite(values) & (values > 0.0)
    )
    plotted = np.where(reliable, values, np.nan)
    if np.any(reliable):
        ax.plot(radius, plotted, label=label, color=color, ls=ls, lw=lw)
    elif int(np.sum(count)) > 0:
        ax.plot(
            [], [], label=f"{label} (sparse; n={int(np.sum(count))})",
            color=color, ls=ls, lw=lw,
        )

def _style_psf_axis(ax, panel_title):
    positive = []
    for line in ax.lines:
        x = np.asarray(line.get_xdata(), dtype=float)
        y = np.asarray(line.get_ydata(), dtype=float)
        good = np.isfinite(x) & (x > 0) & np.isfinite(y) & (y > 0)
        positive.extend(y[good].tolist())
    ax.set_xscale('log')
    ax.set_yscale('log')
    if not positive:
        # No data values are added: limits are placeholders for an empty panel.
        ax.set_xlim(0.01, 10.0)
        ax.set_ylim(1e-8, 1.0)
        ax.text(0.5, 0.5, 'No bins meet the count threshold',
                transform=ax.transAxes, ha='center', va='center')
    ax.set_xlabel('Emission radius r (nm)')
    ax.set_ylabel(r'Radial PSF (electrons nm$^{-2}$ primary$^{-1}$)')
    ax.set_title(panel_title, loc='left', fontweight='bold')
    ax.grid(True, which='both', alpha=0.22)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(frameon=False, fontsize=8)


def plot_radial_psf(
    path,
    profiles,
    masks,
    *,
    n_primaries: int,
    title: str,
    beam_sigma_nm: float = 0.0,
    min_count_per_bin: int = 10,
):
    """Write the four-panel population-resolved radial PSF figure."""
    import matplotlib.pyplot as plt

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    radius = np.asarray(profiles["radius_center_nm"], dtype=float)

    colors = {
        "total": "#161616",
        "single": "#1261a0",
        "strict": "#00a6a6",
        "multiple": "#d1495b",
        "plural": "#e59500",
        "mixed": "#7a5195",
        "other": "#777777",
        "reflection": "#9c27b0",
    }

    min_count_per_bin = int(min_count_per_bin)
    if min_count_per_bin < 1:
        raise ValueError("min_count_per_bin must be positive")

    def curve(ax, name, label, color, ls="-", lw=2.0):
        _plot_psf_curve(
            ax, radius, profiles, name, label, color,
            min_count_per_bin=min_count_per_bin, ls=ls, lw=lw,
        )

    fig, axes = plt.subplots(2, 2, figsize=(13.0, 9.6), constrained_layout=True)
    ax = axes[0, 0]
    curve(ax, "se_total", "All SE (E <= cutoff)", colors["total"], lw=2.4)
    curve(ax, "se_single_inelastic_origin", "Single-inelastic origin", colors["single"])
    curve(ax, "se_strict_single_inelastic", "Strict ballistic subset", colors["strict"], "--", 1.6)
    curve(ax, "se_multiple_origin", "Other / multiple origin", colors["multiple"])
    _style_psf_axis(ax, "(a) Secondary-electron populations")

    ax = axes[0, 1]
    curve(ax, "bse_total", "All BSE (E > cutoff)", colors["total"], lw=2.4)
    curve(ax, "bse_single_elastic", "Exactly 1 elastic, 0 inelastic", colors["single"])
    curve(ax, "bse_plural_elastic_only", ">=2 elastic, 0 inelastic", colors["plural"])
    curve(ax, "bse_elastic_plus_inelastic", "Elastic + inelastic", colors["mixed"])
    curve(ax, "bse_other", "Other BSE", colors["other"], ":", 1.6)
    _style_psf_axis(ax, "(b) Backscattered-electron populations")

    ax = axes[1, 0]
    curve(ax, "transport_total", "All transported emissions", colors["total"], lw=2.4)
    curve(ax, "combined_single", "Combined single", colors["single"])
    curve(ax, "combined_multiple", "Combined multiple", colors["multiple"])
    if np.count_nonzero(masks["direct_barrier_reflection"]):
        curve(ax, "direct_barrier_reflection", "Direct barrier reflection", colors["reflection"], ":", 1.6)
    _style_psf_axis(ax, "(c) Combined populations")

    ax = axes[1, 1]
    for name, label, color, ls in (
        ("se_single_inelastic_origin", "SE single-inelastic origin", colors["single"], "-"),
        ("se_multiple_origin", "SE other / multiple", colors["multiple"], "-"),
        ("bse_single_elastic", "BSE single elastic", colors["strict"], "--"),
        ("bse_plural_elastic_only", "BSE plural elastic only", colors["plural"], "--"),
        ("bse_elastic_plus_inelastic", "BSE elastic + inelastic", colors["mixed"], "-."),
    ):
        count = np.asarray(profiles[f"count__{name}"], dtype=float)
        total = float(np.sum(count))
        if total > 0.0:
            ax.plot(radius, np.cumsum(count) / total, label=label, color=color, ls=ls, lw=1.9)
    ax.set_xscale("log")
    ax.set_xlim(float(radius[0]), float(radius[-1]))
    ax.set_ylim(0.0, 1.02)
    ax.set_xlabel("Emission radius r (nm)")
    ax.set_ylabel("Encircled fraction of population")
    ax.set_title("(d) Population radial extent", loc="left", fontweight="bold")
    ax.grid(True, which="both", alpha=0.22)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(frameon=False, fontsize=8)

    beam_note = "point beam" if float(beam_sigma_nm) == 0.0 else f"Gaussian convolution sigma={float(beam_sigma_nm):g} nm"
    fig.suptitle(
        f"{title}\nN={int(n_primaries):,} incident primaries; {beam_note}; "
        f"PSF bins shown for count >= {min_count_per_bin}",
        fontsize=15,
    )
    fig.savefig(path, dpi=220)
    plt.close(fig)
    return path

def plot_mechanism_detail(
    path,
    profiles,
    *,
    n_primaries: int,
    title: str,
    beam_sigma_nm: float = 0.0,
    min_count_per_bin: int = 10,
):
    """Write refined SE1/SE2, SE-lineage, and BSE-other PSF panels."""
    import matplotlib.pyplot as plt

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    radius = np.asarray(profiles["radius_center_nm"], dtype=float)
    min_count_per_bin = int(min_count_per_bin)
    if min_count_per_bin < 1:
        raise ValueError("min_count_per_bin must be positive")

    colors = {
        "total": "#161616",
        "blue": "#1261a0",
        "red": "#d1495b",
        "orange": "#e59500",
        "purple": "#7a5195",
        "green": "#2a9d8f",
        "gray": "#777777",
    }

    def curve(ax, name, label, color, ls="-", lw=2.0):
        _plot_psf_curve(
            ax, radius, profiles, name, label, color,
            min_count_per_bin=min_count_per_bin, ls=ls, lw=lw,
        )

    fig, axes = plt.subplots(2, 2, figsize=(13.0, 9.6), constrained_layout=True)

    ax = axes[0, 0]
    curve(ax, "se_total", "All SE", colors["total"], lw=2.4)
    curve(ax, "se1_current_rule", "Current SE1", colors["blue"])
    curve(ax, "se2_current_rule", "Current SE2", colors["red"])
    _style_psf_axis(ax, "(a) Current imaging SE1/SE2")

    ax = axes[0, 1]
    curve(ax, "se_multiple_origin", "All multiple-origin SE", colors["total"], lw=2.4)
    curve(
        ax, "se_mult_gen1_after_elastic", "Generation 1 after parent elastic",
        colors["orange"],
    )
    curve(
        ax, "se_mult_gen1_after_prior_inelastic",
        "Generation 1 after prior inelastic", colors["green"], "--",
    )
    curve(
        ax, "se_mult_generation_2plus", "Generation >=2",
        colors["purple"],
    )
    curve(ax, "se_mult_primary", "Low-energy primary", colors["gray"], ":", 1.6)
    _style_psf_axis(ax, "(b) Multiple-origin SE decomposition")

    ax = axes[1, 0]
    curve(ax, "bse_other", "All other BSE", colors["total"], lw=2.4)
    curve(
        ax, "bse_other_high_energy_cascade", "High-energy cascade",
        colors["purple"],
    )
    curve(
        ax, "bse_other_primary_inelastic_only", "Primary, inelastic only",
        colors["orange"], "--",
    )
    curve(
        ax, "bse_other_primary_no_collision", "Primary, no bulk collision",
        colors["gray"], ":", 1.6,
    )
    curve(
        ax, "bse_other_unclassified", "Unclassified residual",
        colors["red"], ":", 1.6,
    )
    _style_psf_axis(ax, "(c) Other-BSE decomposition")

    ax = axes[1, 1]
    for name, label, color, ls in (
        ("se1_current_rule", "Current SE1", colors["blue"], "-"),
        ("se2_current_rule", "Current SE2", colors["red"], "-"),
        (
            "se_single_inelastic_origin", "Single-inelastic origin",
            colors["green"], "--",
        ),
        ("se_multiple_origin", "Multiple origin", colors["purple"], "--"),
    ):
        count = np.asarray(profiles[f"count__{name}"], dtype=float)
        total = float(np.sum(count))
        if total > 0.0:
            ax.plot(
                radius, np.cumsum(count) / total, label=label,
                color=color, ls=ls, lw=1.9,
            )
    ax.set_xscale("log")
    ax.set_xlim(float(radius[0]), float(radius[-1]))
    ax.set_ylim(0.0, 1.02)
    ax.set_xlabel("Emission radius r (nm)")
    ax.set_ylabel("Encircled fraction of population")
    ax.set_title("(d) Current-rule versus origin-rule extent", loc="left", fontweight="bold")
    ax.grid(True, which="both", alpha=0.22)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(frameon=False, fontsize=8)

    beam_note = "point beam" if float(beam_sigma_nm) == 0.0 else f"Gaussian convolution sigma={float(beam_sigma_nm):g} nm"
    fig.suptitle(
        f"{title}\nN={int(n_primaries):,}; {beam_note}; "
        f"PSF bins shown for count >= {min_count_per_bin}",
        fontsize=15,
    )
    fig.savefig(path, dpi=220)
    plt.close(fig)
    return path

def plot_seed_radius_depth(
    path,
    data,
    masks,
    *,
    n_primaries: int,
    title: str,
    n_bins: int = 90,
):
    """Write four seed-radius-versus-depth density panels for SE populations."""
    import matplotlib.pyplot as plt
    from matplotlib.colors import LogNorm

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    n_bins = int(n_bins)
    if n_bins < 20:
        raise ValueError("seed density requires at least 20 bins")

    birth_radius, birth_depth = birth_radius_depth_nm(data)
    se_mask = np.asarray(masks["se_total"], dtype=bool)
    if not np.any(se_mask):
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.text(0.5, 0.5, "No emitted SEs: seed density is empty", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        fig.suptitle(title)
        fig.savefig(path, dpi=220)
        plt.close(fig)
        return path

    radius_scale = 0.01
    depth_scale = 0.01

    def forward(values, scale):
        return np.log10(1.0 + np.asarray(values, dtype=float) / scale)

    u_all = forward(birth_radius[se_mask], radius_scale)
    v_all = forward(birth_depth[se_mask], depth_scale)
    u_edges = np.linspace(0.0, max(float(np.max(u_all)) * 1.001, 1.0), n_bins + 1)
    v_edges = np.linspace(0.0, max(float(np.max(v_all)) * 1.001, 1.0), n_bins + 1)

    panels = (
        ("se_single_inelastic_origin", "(a) Single-inelastic-origin SE"),
        ("se_multiple_origin", "(b) Multiple-origin SE"),
        ("se1_current_rule", "(c) Current imaging SE1"),
        ("se2_current_rule", "(d) Current imaging SE2"),
    )
    histograms = []
    maximum = 1
    for name, _ in panels:
        mask = np.asarray(masks[name], dtype=bool)
        hist, _, _ = np.histogram2d(
            forward(birth_radius[mask], radius_scale),
            forward(birth_depth[mask], depth_scale),
            bins=(u_edges, v_edges),
        )
        histograms.append(hist)
        maximum = max(maximum, int(np.max(hist)))

    fig, axes = plt.subplots(2, 2, figsize=(13.0, 9.6), constrained_layout=True)
    mesh = None
    for ax, (name, panel_title), hist in zip(axes.ravel(), panels, histograms):
        masked = np.ma.masked_less_equal(hist.T, 0.0)
        mesh = ax.pcolormesh(
            u_edges, v_edges, masked, shading="auto", cmap="magma",
            norm=LogNorm(vmin=1.0, vmax=float(max(2, maximum))), rasterized=True,
        )
        mask = np.asarray(masks[name], dtype=bool)
        count = int(np.count_nonzero(mask))
        shallow_far = (
            float(np.mean((birth_depth[mask] <= 2.0) & (birth_radius[mask] >= 100.0)))
            if count else float("nan")
        )
        ax.axvline(forward(100.0, radius_scale), color="white", ls="--", lw=1.0, alpha=0.8)
        ax.axhline(forward(2.0, depth_scale), color="white", ls="--", lw=1.0, alpha=0.8)
        ax.set_title(panel_title, loc="left", fontweight="bold")
        ax.text(
            0.98, 0.96,
            f"n={count:,}\nshallow and far={100.0 * shallow_far:.1f}%" if count else "n=0",
            transform=ax.transAxes, ha="right", va="top", color="white", fontsize=8,
            bbox={"facecolor": "black", "alpha": 0.45, "edgecolor": "none", "pad": 3},
        )
        radius_ticks = [0.0, 0.1, 1.0, 10.0, 100.0, 1000.0]
        depth_ticks = [0.0, 0.1, 1.0, 10.0, 100.0]
        radius_ticks = [value for value in radius_ticks if value <= float(np.max(birth_radius[se_mask])) * 1.05]
        depth_ticks = [value for value in depth_ticks if value <= float(np.max(birth_depth[se_mask])) * 1.05]
        ax.set_xticks([forward(value, radius_scale) for value in radius_ticks])
        ax.set_xticklabels([f"{value:g}" for value in radius_ticks])
        ax.set_yticks([forward(value, depth_scale) for value in depth_ticks])
        ax.set_yticklabels([f"{value:g}" for value in depth_ticks])
        ax.set_xlabel("Birth radius from impact (nm)")
        ax.set_ylabel("Birth depth (nm)")

    if mesh is not None:
        colorbar = fig.colorbar(mesh, ax=axes.ravel().tolist(), shrink=0.88, pad=0.02)
        colorbar.set_label("Emissions per bin")
    fig.suptitle(
        f"{title}\nN={int(n_primaries):,}; intrinsic point-response birth coordinates; "
        "log-like axes include zero\nDashed guides: radius 100 nm, depth 2 nm",
        fontsize=14,
    )
    fig.savefig(path, dpi=220)
    plt.close(fig)
    return path


## 4. Initialize Taichi and load Au transport tables


In [ ]:
if analyze_existing:
    print('Re-analysis mode: skipping device initialization and material loading.')
else:
    if n < 1 or batch_size < 1:
        raise ValueError("n and batch_size must be positive")
    if capacity < min(n, batch_size):
        raise ValueError("capacity must be >= the number of primaries in one batch")
    if energy_ev <= 0.0:
        raise ValueError("energy_ev must be positive")
    if not 0.0 <= angle_deg < 90.0:
        raise ValueError("angle_deg must lie in [0, 90)")
    if steps_per_chunk < 1:
        raise ValueError("steps_per_chunk must be positive")

    if not database.is_file():
        raise FileNotFoundError(f'Material database not found: {database}')
    if precision not in ('f32', 'f64'):
        raise ValueError("precision must be 'f32' or 'f64'")
    if arch == 'metal' and precision == 'f64':
        raise ValueError('Metal requires f32 on the supported Taichi build')
    if seed_bins < 20 or min_count_per_bin < 1:
        raise ValueError('seed_bins must be >= 20 and min_count_per_bin >= 1')

    cfg = BackendConfig(
        arch=arch, max_particles=capacity, max_emissions=capacity,
        default_fp=precision, default_ip='i32', random_seed=seed,
    )
    cfg.validate()
    backend = dict(cpu=ti.cpu, cuda=ti.cuda, metal=ti.metal,
                   vulkan=ti.vulkan, gpu=ti.gpu, opengl=ti.opengl)[arch]
    fp = ti.f32 if precision == 'f32' else ti.f64
    ti.init(
        arch=backend,
        default_fp=fp,
        default_ip=ti.i32,
        fast_math=fast_math,
        enable_fallback=enable_fallback,
        offline_cache=True,
        random_seed=seed,
    )

    sample, host = load_reference_plane_tables(str(database), material)
    bulk = BulkPhysicsConfig.from_sample(sample)
    surface = SurfacePhysicsConfig.from_sample(sample)
    simulation_cutoff = float(getattr(sample.cfg, 'bse_cutoff_ev', 50.0))
    if cutoff_ev is not None:
        simulation_cutoff = float(cutoff_ev)
    if not math.isfinite(simulation_cutoff) or simulation_cutoff <= 0:
        raise ValueError('The SE/BSE cutoff must be finite and positive')
    engine = PlaneTransportEngine(
        ti, fp, host, bulk, surface, capacity, bse_cutoff_ev=simulation_cutoff,
    )

    # Freeze the executed simulation settings for later metadata and analysis.
    run_settings = dict(
        database=str(database), material=material, energy_ev=energy_ev,
        angle_deg=angle_deg, arch=arch, precision=precision, seed=seed,
        n=n, batch_size=batch_size, capacity=capacity,
        steps_per_chunk=steps_per_chunk, fast_math=fast_math,
        enable_fallback=enable_fallback,
    )
    print(f'Material: {material}; energy: {energy_ev:g} eV; incidence: {angle_deg:g}°')
    print(f'Backend / precision: {arch} / {precision}; cutoff: {simulation_cutoff:g} eV')


## 5. Warm up kernels and transport the primaries

Warm-up primaries are excluded from the result. Each batch checks particle/emission overflow and transport safety limits. Rerun initialization before repeating this cell for a fresh seeded run.


In [ ]:
if analyze_existing:
    print('Re-analysis mode: skipping transport.')
else:
    current_settings = dict(
        database=str(database), material=material, energy_ev=energy_ev,
        angle_deg=angle_deg, arch=arch, precision=precision, seed=seed,
        n=n, batch_size=batch_size, capacity=capacity,
        steps_per_chunk=steps_per_chunk, fast_math=fast_math,
        enable_fallback=enable_fallback,
    )
    if current_settings != run_settings:
        raise RuntimeError('Simulation settings changed. Rerun the initialization cell first.')
    warm_n = min(64, batch_size, n)
    engine.run_plane(
        n=warm_n,
        energy_vac_ev=energy_ev,
        alpha_deg=angle_deg,
        steps_per_chunk=min(32, steps_per_chunk),
        copy_emissions=False,
    )
    ti.sync()

    accum = {key: [] for key in EVENT_KEYS}
    batch_index_parts = []
    diagnostic_sum = None
    primary_offset = 0
    particle_offset = 0
    elapsed_total = 0.0
    n_batches = int(math.ceil(n / batch_size))

    print(f"material: {material}; energy: {energy_ev:g} eV; angle: {angle_deg:g} deg")
    print(f"arch / precision: {arch} / {precision}")
    print(f"primaries: {n:,} in {n_batches} batch(es); capacity per batch: {capacity:,}")
    for batch in range(n_batches):
        n_batch = min(batch_size, n - primary_offset)
        start = time.perf_counter()
        result = engine.run_plane(
            n=n_batch,
            energy_vac_ev=energy_ev,
            alpha_deg=angle_deg,
            steps_per_chunk=steps_per_chunk,
            copy_emissions=True,
        )
        ti.sync()
        elapsed = time.perf_counter() - start
        elapsed_total += elapsed
        diag = result["diagnostics"]
        if result["overflow"] or result["emission_overflow"] or diag["emission_buffer_overflow"]:
            raise RuntimeError(
                f"batch {batch + 1} overflowed; reduce batch_size or increase capacity"
            )
        if diag["step_limit_hit"] or diag["generation_limit_hit"]:
            raise RuntimeError(
                f"batch {batch + 1} hit a transport safety limit: {diag}"
            )

        if diagnostic_sum is None:
            diagnostic_sum = {name: 0 for name in diag}
        for name, value in diag.items():
            diagnostic_sum[name] += int(value)

        local = {key: np.asarray(result[key]).copy() for key in EVENT_KEYS}
        local["emission_root_primary_id"] += primary_offset
        local["emission_electron_id"] += particle_offset
        parent = local["emission_parent_id"]
        parent[parent >= 0] += particle_offset
        for key in EVENT_KEYS:
            accum[key].append(local[key])
        batch_index_parts.append(
            np.full(local["emission_energy_ev"].size, batch, dtype=np.int32)
        )

        primary_offset += n_batch
        particle_offset += int(result["allocated"])
        print(
            f"[{batch + 1}/{n_batches}] n={n_batch:,}, emissions={local['emission_energy_ev'].size:,}, "
            f"allocated={int(result['allocated']):,}, elapsed={elapsed:.3f}s"
        )

    merged = {
        key: (
            np.concatenate(parts)
            if parts
            else np.asarray([], dtype=np.float64)
        )
        for key, parts in accum.items()
    }
    merged["emission_batch_index"] = (
        np.concatenate(batch_index_parts)
        if batch_index_parts
        else np.asarray([], dtype=np.int32)
    )
    n_emit = int(merged["emission_energy_ev"].size)
    if diagnostic_sum is None or n_emit != diagnostic_sum["escapes"]:
        raise RuntimeError("merged emission count does not match accumulated escape count")


## 6. Save raw emission events


In [ ]:
if analyze_existing:
    if not existing_events.is_file():
        raise FileNotFoundError(f'Saved events not found: {existing_events.resolve()}')
    with np.load(existing_events, allow_pickle=False) as archive:
        data = {name: archive[name] for name in archive.files}
    print(f'Loaded {existing_events.resolve()} for re-analysis; raw archive is unchanged.')
else:
    prefix = Path(output_prefix)
    prefix.parent.mkdir(parents=True, exist_ok=True)
    events_path = prefix.with_name(prefix.name + '_events.npz')
    metadata = {
        "schema": "seemc-plane-psf-events-v1",
        "seemc_taichi_version": __version__,
        "material": str(run_settings["material"]),
        "material_kind": "metal" if sample.is_metal else "nonconductor",
        "energy_vac_ev": float(run_settings["energy_ev"]),
        "angle_deg": float(run_settings["angle_deg"]),
        "n_primaries": int(run_settings["n"]),
        "batch_size": int(run_settings["batch_size"]),
        "n_batches": n_batches,
        "capacity_per_batch": int(run_settings["capacity"]),
        "arch": str(run_settings["arch"]),
        "precision": str(run_settings["precision"]),
        "seed": int(run_settings["seed"]),
        "steps_per_chunk": int(run_settings["steps_per_chunk"]),
        "bse_cutoff_ev": simulation_cutoff,
        "point_beam": True,
        "position_unit": "angstrom",
        "surface_convention": "solid z>0; vacuum z<0; emission surface z=0",
        "bulk_physics": asdict(bulk),
        "surface_physics": asdict(surface),
        "diagnostics": diagnostic_sum,
        "classification_descriptions": CLASSIFICATION_DESCRIPTIONS,
        "elapsed_s": elapsed_total,
        "fast_math": run_settings["fast_math"],
        "enable_fallback": run_settings["enable_fallback"],
        "primaries_per_s": run_settings["n"] / elapsed_total if elapsed_total > 0.0 else None,
    }
    _save_archive_atomic(
        events_path,
        metadata_json=np.asarray(json.dumps(metadata, sort_keys=True)),
        n_primaries=np.asarray(run_settings["n"], dtype=np.int64),
        bse_cutoff_ev=np.asarray(simulation_cutoff, dtype=float),
        **merged,
    )
    print(f"wrote {events_path}")

    data = merged | {
        'metadata_json': np.asarray(json.dumps(metadata, sort_keys=True)),
        'n_primaries': np.asarray(metadata['n_primaries'], dtype=np.int64),
        'bse_cutoff_ev': np.asarray(simulation_cutoff, dtype=float),
    }
    print(f'Saved {n_emit:,} emissions from {metadata["n_primaries"]:,} primaries')


## 7. Analyze populations and generate PSF plots

Change the analysis settings in cell 2 and rerun from here to replot without new transport. Gaussian convolution affects the plotted emission coordinates; birth-radius/depth plots use the original creation coordinates. All raw events remain in the NPZ archive.


In [ ]:
prefix = Path(output_prefix)
prefix.parent.mkdir(parents=True, exist_ok=True)
metadata = json.loads(str(np.asarray(data['metadata_json']).item()))
n_primaries = int(np.asarray(data.get("n_primaries", metadata.get("n_primaries"))).item())
saved_cutoff = float(
    np.asarray(data.get("bse_cutoff_ev", metadata.get("bse_cutoff_ev", 50.0))).item()
)
cutoff = saved_cutoff if cutoff_ev is None else float(cutoff_ev)
if not math.isfinite(cutoff) or cutoff <= 0.0:
    raise ValueError("cutoff must be finite and positive")
if min_count_per_bin < 1:
    raise ValueError("min_count_per_bin must be positive")
if seed_bins < 20:
    raise ValueError("seed_bins must be at least 20")

masks = classify_emissions(data, cutoff_ev=cutoff)
x_nm, y_nm = emission_xy_nm(
    data, beam_sigma_nm=beam_sigma_nm, seed=convolution_seed
)
radius_nm = np.hypot(x_nm, y_nm)
if r_max_nm is None:
    observed = float(np.max(radius_nm)) if radius_nm.size else r_min_nm * 10.0
    r_max = max(r_min_nm * 10.0, observed * 1.01)
else:
    r_max = float(r_max_nm)
edges = radial_bin_edges(r_min_nm, r_max, radial_bins)
profiles = radial_profiles(radius_nm, masks, edges, n_primaries)
rows = population_summary(data, radius_nm, masks, n_primaries)
seed_rows = seed_radius_depth_summary(data, masks)

paths = {
    'summary': prefix.with_name(prefix.name + '_population_summary.csv'),
    'profile': prefix.with_name(prefix.name + '_radial_psf.csv'),
    'plot': prefix.with_name(prefix.name + '_radial_psf.png'),
    'detail_plot': prefix.with_name(prefix.name + '_mechanism_detail.png'),
    'seed_summary': prefix.with_name(prefix.name + '_seed_radius_depth_summary.csv'),
    'seed_plot': prefix.with_name(prefix.name + '_seed_radius_depth.png'),
}
write_rows_csv(paths["summary"], rows)
write_profile_csv(paths["profile"], profiles)
write_rows_csv(paths["seed_summary"], seed_rows)
material = str(metadata.get("material", "material"))
energy = float(metadata.get("energy_vac_ev", metadata.get("energy_ev", float("nan"))))
angle = float(metadata.get("angle_deg", float("nan")))
title = f"{material}, {energy:g} eV, incidence {angle:g} deg; cutoff {cutoff:g} eV"
plot_radial_psf(
    paths["plot"], profiles, masks,
    n_primaries=n_primaries,
    title=title,
    beam_sigma_nm=beam_sigma_nm,
    min_count_per_bin=min_count_per_bin,
)
plot_mechanism_detail(
    paths["detail_plot"], profiles,
    n_primaries=n_primaries,
    title=title,
    beam_sigma_nm=beam_sigma_nm,
    min_count_per_bin=min_count_per_bin,
)
plot_seed_radius_depth(
    paths["seed_plot"], data, masks,
    n_primaries=n_primaries,
    title=title,
    n_bins=seed_bins,
)

lookup = {row["population"]: row for row in rows}
print(f"analysis cutoff: {cutoff:g} eV")
print(f"beam convolution sigma: {float(beam_sigma_nm):g} nm")
print(f"observed maximum radius: {(float(np.max(radius_nm)) if radius_nm.size else 0.0):.6g} nm")
for name in (
    "se_total", "se_single_inelastic_origin", "se_multiple_origin",
    "se_mult_gen1_after_elastic", "se_mult_gen1_after_prior_inelastic",
    "se_mult_generation_2plus",
    "se1_current_rule", "se2_current_rule",
    "bse_total", "bse_single_elastic", "bse_plural_elastic_only",
    "bse_elastic_plus_inelastic", "bse_other_high_energy_cascade",
    "bse_other_primary_inelastic_only", "combined_single", "combined_multiple",
):
    row = lookup[name]
    print(
        f"  {name}: count={row['count']:,}, "
        f"yield={row['yield_per_primary']:.8g}, r90={row['r90_nm']:.6g} nm"
    )
print(f"wrote {paths['summary']}")
print(f"wrote {paths['profile']}")
print(f"wrote {paths['plot']}")
print(f"wrote {paths['detail_plot']}")
print(f"wrote {paths['seed_summary']}")
print(f"wrote {paths['seed_plot']}")


## 8. Inspect the summary and plots in the notebook


In [ ]:
selected = ['se_total', 'se1_current_rule', 'se2_current_rule',
            'bse_total', 'bse_single_elastic', 'bse_plural_elastic_only',
            'bse_elastic_plus_inelastic']
summary_lookup = {row['population']: row for row in rows}
table = ['| Population | Count | Yield per primary | r90 (nm) |',
         '|---|---:|---:|---:|']
for name in selected:
    row = summary_lookup[name]
    table.append(f"| {name} | {row['count']:,} | {row['yield_per_primary']:.6g} | {row['r90_nm']:.6g} |")
display(Markdown('\n'.join(table)))
for key in ('plot', 'detail_plot', 'seed_plot'):
    display(Image(filename=str(paths[key])))


The notebook keeps `data` (raw events), `metadata`, `masks`, `profiles`, `rows`, and `seed_rows` available for further analysis. Saved outputs include the raw events NPZ, population and seed summary CSVs, radial PSF CSV, and three PNG figures. The NPZ schema is compatible with the existing PSF re-analysis tools.

The notebook has not been run against your cluster GPU or Au material database here. Its batching, collision-history classification, and plots use the same implementation as the saved v0.9.3 PSF workflow.
